# 10 — Human-in-the-Loop: Confidence-Based Review Workflow

**Time**: ~15 minutes · **Pattern**: Confidence scoring + review queue

---

## Insurance Scenario

When processing insurance claims automatically, not every extraction will be 100% accurate. For an insurance company, incorrect data has real consequences:

- Wrong claim amounts → overpayments or underpayments
- Wrong policyholder details → compliance violations
- Missed information → delayed claim resolution

The solution: **Use confidence scores to automatically route uncertain extractions to human reviewers.**

### The Pattern

```
Document → AI Extraction → Confidence Check
                               ├─ High confidence (≥0.90)  → Auto-accept
                               ├─ Medium confidence (0.70–0.90) → Flag for review
                               └─ Low confidence (<0.70)   → Manual entry required
```

---

## Prerequisites

Complete [00-setup-and-basics.ipynb](00-setup-and-basics.ipynb) first.

In [ ]:
import os
import json
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, AnalyzeResult
import pandas as pd
from datetime import datetime

load_dotenv(dotenv_path=os.path.join("..", ".env"))

_api_key = os.environ.get("DOCUMENTINTELLIGENCE_API_KEY", "")
_credential = AzureKeyCredential(_api_key) if _api_key else DefaultAzureCredential()

client = DocumentIntelligenceClient(
    endpoint=os.environ["DOCUMENTINTELLIGENCE_ENDPOINT"],
    credential=_credential
)
print("✅ Client ready")

## Step 1: Define Confidence Thresholds

Different fields may require different confidence thresholds depending on their business impact.

In [ ]:
# Confidence thresholds — customize for your risk tolerance
THRESHOLDS = {
    # High-impact financial fields — require higher confidence
    "InvoiceTotal":   {"auto_accept": 0.95, "review": 0.80},
    "AmountDue":      {"auto_accept": 0.95, "review": 0.80},
    "SubTotal":       {"auto_accept": 0.95, "review": 0.80},
    
    # Identity fields — stricter for compliance
    "VendorName":     {"auto_accept": 0.90, "review": 0.75},
    "CustomerName":   {"auto_accept": 0.90, "review": 0.75},
    
    # Standard fields — default thresholds
    "DEFAULT":        {"auto_accept": 0.90, "review": 0.70},
}

def classify_confidence(field_name, confidence):
    """Classify a field's confidence into auto-accept, review, or manual."""
    thresholds = THRESHOLDS.get(field_name, THRESHOLDS["DEFAULT"])
    if confidence >= thresholds["auto_accept"]:
        return "auto_accept"
    elif confidence >= thresholds["review"]:
        return "review"
    else:
        return "manual_entry"

print("Confidence thresholds configured:")
for field, thresh in THRESHOLDS.items():
    print(f"  {field:20s}  auto-accept ≥ {thresh['auto_accept']:.0%}, review ≥ {thresh['review']:.0%}")

## Step 2: Analyze a Document and Score Fields

In [ ]:
# Analyze a sample invoice
invoice_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-invoice.pdf"

poller = client.begin_analyze_document(
    "prebuilt-invoice",
    AnalyzeDocumentRequest(url_source=invoice_url)
)
result: AnalyzeResult = poller.result()

print(f"Document analyzed. Processing {len(result.documents)} invoice(s)...")

In [ ]:
# Score each field and build a review dashboard
review_items = []

if result.documents:
    invoice = result.documents[0]
    
    for field_name, field in invoice.fields.items():
        if field_name == "Items":  # Skip array fields
            continue
        
        confidence = field.get("confidence", 0)
        value = field.get("content", "N/A")
        status = classify_confidence(field_name, confidence)
        
        review_items.append({
            "Field": field_name,
            "Extracted Value": str(value)[:40],
            "Confidence": confidence,
            "Status": status,
            "Icon": {"auto_accept": "✅", "review": "⚠️", "manual_entry": "❌"}.get(status),
        })

df = pd.DataFrame(review_items)
df = df.sort_values("Confidence", ascending=True)

print("\nFIELD-LEVEL REVIEW DASHBOARD")
print("=" * 80)
print(df[["Icon", "Field", "Extracted Value", "Confidence", "Status"]].to_string(index=False))

## Step 3: Generate a Review Queue

Filter out only the fields that need human attention.

In [ ]:
# Fields needing human review
needs_review = df[df["Status"].isin(["review", "manual_entry"])]
auto_accepted = df[df["Status"] == "auto_accept"]

print(f"📊 PROCESSING SUMMARY")
print(f"{'=' * 50}")
print(f"  ✅ Auto-accepted:    {len(auto_accepted)} fields")
print(f"  ⚠️ Needs review:     {len(needs_review[needs_review['Status'] == 'review'])} fields")
print(f"  ❌ Manual entry:     {len(needs_review[needs_review['Status'] == 'manual_entry'])} fields")
print()

if not needs_review.empty:
    print("ITEMS FOR HUMAN REVIEW:")
    print("-" * 50)
    for _, row in needs_review.iterrows():
        print(f"  {row['Icon']} {row['Field']}")
        print(f"    AI extracted: \"{row['Extracted Value']}\"")
        print(f"    Confidence: {row['Confidence']:.2%}")
        print(f"    Action: {'Verify and correct' if row['Status'] == 'review' else 'Enter manually'}")
        print()
else:
    print("🎉 All fields passed confidence checks — no human review needed!")

## Step 4: Build a Review Record

Create a structured review record that a human reviewer can process. This could be sent to a review UI, a ticketing system (ServiceNow, Jira), or a simple queue.

In [ ]:
# Build a review record
review_record = {
    "document_id": poller.details.get("operation_id", "unknown"),
    "model_used": result.model_id,
    "timestamp": datetime.now().isoformat(),
    "overall_status": "needs_review" if not needs_review.empty else "auto_accepted",
    "auto_accepted_fields": {},
    "review_required_fields": {},
}

for _, row in df.iterrows():
    field_data = {
        "ai_value": row["Extracted Value"],
        "confidence": row["Confidence"],
        "human_corrected_value": None,  # To be filled by reviewer
    }
    if row["Status"] == "auto_accept":
        review_record["auto_accepted_fields"][row["Field"]] = field_data
    else:
        field_data["review_status"] = "pending"
        review_record["review_required_fields"][row["Field"]] = field_data

print("REVIEW RECORD:")
print(json.dumps(review_record, indent=2, default=str))

## Step 5: Simulate Human Review & Correction

In a production system, this step would be a UI. Here we simulate a reviewer correcting a field.

In [ ]:
# Simulate a human reviewer correcting fields
def simulate_human_review(review_record):
    """Simulate human corrections on flagged fields."""
    reviewed = review_record.copy()
    
    for field_name, field_data in reviewed["review_required_fields"].items():
        # In production, these corrections come from a reviewer UI
        # Here we simulate accepting the AI value (common case)
        field_data["human_corrected_value"] = field_data["ai_value"]
        field_data["review_status"] = "approved"
        field_data["reviewer"] = "adjuster_jane_doe"
        field_data["review_timestamp"] = datetime.now().isoformat()
    
    reviewed["overall_status"] = "completed"
    return reviewed

completed_record = simulate_human_review(review_record)

print("COMPLETED REVIEW RECORD:")
print(json.dumps(completed_record, indent=2, default=str))

## Step 6: Feed Corrections Back for Model Improvement

Human corrections are valuable training data. Track them to improve your models over time.

In [ ]:
def track_corrections(completed_record):
    """Track divergence between AI and human values for model improvement."""
    corrections = []
    
    for field_name, field_data in completed_record["review_required_fields"].items():
        ai_val = field_data["ai_value"]
        human_val = field_data["human_corrected_value"]
        
        corrections.append({
            "field": field_name,
            "ai_value": ai_val,
            "human_value": human_val,
            "was_corrected": ai_val != human_val,
            "original_confidence": field_data["confidence"],
        })
    
    return corrections

corrections = track_corrections(completed_record)

print("CORRECTION TRACKING:")
print("=" * 60)
df_corrections = pd.DataFrame(corrections)
if not df_corrections.empty:
    print(df_corrections.to_string(index=False))
    
    corrected_count = sum(1 for c in corrections if c["was_corrected"])
    total = len(corrections)
    print(f"\n📊 AI accuracy on reviewed fields: {(total - corrected_count) / total:.0%}")
    print(f"   Fields corrected: {corrected_count} / {total}")
    print()
    print("💡 Tip: Use corrected data to retrain custom models for better accuracy.")
    print("   See notebook 07 for custom model training.")
else:
    print("No corrections tracked.")

## Complete Workflow: End-to-End Function

Here's the full pattern wrapped in a reusable function.

In [ ]:
def process_insurance_document(client, document_url, model_id="prebuilt-invoice", thresholds=None):
    """
    Process an insurance document with confidence-based human review.
    
    Returns:
        dict with 'auto_accepted', 'needs_review', and 'all_fields' keys.
    """
    if thresholds is None:
        thresholds = {"auto_accept": 0.90, "review": 0.70}
    
    # Analyze
    poller = client.begin_analyze_document(
        model_id,
        AnalyzeDocumentRequest(url_source=document_url)
    )
    result = poller.result()
    
    auto_accepted = {}
    needs_review = {}
    
    if result.documents:
        for doc in result.documents:
            for name, field in (doc.fields or {}).items():
                if name == "Items":
                    continue
                conf = field.get("confidence", 0)
                value = field.get("content")
                
                entry = {"value": value, "confidence": conf}
                
                if conf >= thresholds["auto_accept"]:
                    auto_accepted[name] = entry
                else:
                    needs_review[name] = entry
    
    return {
        "auto_accepted": auto_accepted,
        "needs_review": needs_review,
        "all_fields": {**auto_accepted, **needs_review},
        "requires_human_review": len(needs_review) > 0,
        "model_id": result.model_id,
    }

# Usage example
result = process_insurance_document(
    client,
    "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-invoice.pdf"
)

print(f"Auto-accepted: {len(result['auto_accepted'])} fields")
print(f"Needs review:  {len(result['needs_review'])} fields")
print(f"Requires human review: {result['requires_human_review']}")

---

## Architecture for Production

```mermaid
flowchart TD
    A[Document Upload] --> B[Document Intelligence API]
    B --> C{Confidence Check}
    C -->|≥ 0.90| D[Auto-Accept → Claims System]
    C -->|0.70 - 0.90| E[Review Queue]
    C -->|< 0.70| F[Manual Entry Queue]
    E --> G[Human Reviewer]
    F --> G
    G --> H[Approved Data → Claims System]
    G --> I[Correction Log]
    I --> J[Retrain Custom Model]
```

### Integration Options for the Review Queue

| Option | Best For |
|---|---|
| **Azure Queue Storage** | Simple queue-based routing |
| **Power Automate** | No-code approval workflows |
| **Custom Web App** | Full-featured review UI with field-level editing |
| **ServiceNow / Jira** | Enterprise ticketing integration |
| **Azure Logic Apps** | Event-driven workflows with notifications |

## Key Takeaways

| Feature | Value for Insurance |
|---|---|
| **Confidence thresholds** | Customize risk tolerance per field type |
| **Auto-accept** | Speed up processing for high-confidence extractions |
| **Review queue** | Ensure accuracy for critical financial and identity fields |
| **Correction tracking** | Build a feedback loop to improve models over time |

## Recommended Thresholds for Insurance

| Field Category | Auto-Accept | Review | Manual |
|---|---|---|---|
| Financial (amounts, totals) | ≥ 0.95 | 0.80–0.95 | < 0.80 |
| Identity (names, IDs) | ≥ 0.90 | 0.75–0.90 | < 0.75 |
| Dates | ≥ 0.90 | 0.70–0.90 | < 0.70 |
| Addresses | ≥ 0.85 | 0.65–0.85 | < 0.65 |

## Next Steps

| Notebook | What You'll Learn |
|---|---|
| [11-batch-processing.ipynb](11-batch-processing.ipynb) | Process multiple documents at scale |
| [07-custom-model.ipynb](07-custom-model.ipynb) | Train a custom model with corrected data |